Lecture 7 — Class Exercise
Part-to-Whole: Hierarchical Visualization
Push to: week07/lecture07_exercise.ipynb

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


Task 1 — Treemap: fossil fuel dependency by country
What to build: A treemap showing fossil fuel TWh only, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

Requirements:

Filter to fossil sources only before plotting
Use path=['Region', 'Country', 'Source'] for the hierarchy
Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
Show TWh values in labels — no percentages
Grey out parent nodes (Region and Country level)
Insight title naming which region or country is most fossil-dependent
💡 df.loc[df['Source_Type'] == 'Fossil']

In [2]:
# Task 1 — Fossil-fuel treemap
df_fossil = df.loc[df["Source_Type"] == "Fossil"].copy()

# Rank regions and countries by total fossil energy (TWh)
region_fossil = df_fossil.groupby("Region")["TWh"].sum().sort_values(ascending=False)
country_fossil = df_fossil.groupby("Country")["TWh"].sum().sort_values(ascending=False)

top_region = region_fossil.index[0]
top_region_twh = region_fossil.iloc[0]
top_country = country_fossil.index[0]
top_country_twh = country_fossil.iloc[0]

# CVD-safe course palette: coal blue, oil orange, natural gas yellow
fossil_colors = {
    "Coal": "#2E75B6",
    "Oil": "#E07B39",
    "Natural Gas": "#F5C842",
}

fig = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map=fossil_colors,
    labels={"TWh": "Energy (TWh)", "Source": "Fossil source"},
    title=(
        f"{top_region} has the largest fossil-energy total "
        f"({top_region_twh:,.0f} TWh); {top_country} leads among countries "
        f"({top_country_twh:,.0f} TWh)"
    ),
    template="plotly_white",
    height=750,
)
# Show labels and absolute TWh values only (no percentages).
fig.update_traces(
    textinfo="label+value",
    texttemplate="%{label}<br>%{value:,.0f} TWh",
    hovertemplate="<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>",
)

# Keep source leaves coloured; grey out Region and Country parent nodes.
trace = fig.data[0]
trace.marker.colors = [
    fossil_colors.get(label, "#DDDDDD") if parent else "#DDDDDD"
    for label, parent in zip(trace.labels, trace.parents)
]
fig.update_layout(font=dict(family="Arial", size=12))
fig.show()

print(f"Largest fossil-energy region: {top_region} ({top_region_twh:,.1f} TWh)")
print(f"Largest fossil-energy country: {top_country} ({top_country_twh:,.1f} TWh)")

Largest fossil-energy region: Asia (35,045.1 TWh)
Largest fossil-energy country: Saudi Arabia (9,671.0 TWh)


Task 2 — Treemap vs bar: low-carbon energy by country
What to build: Build both a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

Requirements:

Filter to Source_Type == 'Low-carbon' and aggregate TWh by country
Treemap: single-level path=['All', 'Country'] with a dummy root node labelled 'Low-carbon'
Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
Both charts show TWh values, not percentages
Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — Low-carbon treemap and horizontal bar chart
df_low_carbon = df.loc[df["Source_Type"] == "Low-carbon"].copy()

low_carbon_by_country = (
    df_low_carbon.groupby("Country", as_index=False)["TWh"]
    .sum()
    .sort_values("TWh", ascending=False)
)

# Add the required dummy root node for the treemap.
low_carbon_by_country["All"] = "Low-carbon"
leading_country = low_carbon_by_country.iloc[0]["Country"]
leading_twh = low_carbon_by_country.iloc[0]["TWh"]

# Treemap: one hierarchy level below the Low-carbon root.
fig_treemap = px.treemap(
    low_carbon_by_country,
    path=["All", "Country"],
    values="TWh",
    color_discrete_sequence=["#2E75B6"],
    labels={"TWh": "Energy (TWh)"},
    title="Low-carbon energy by country (Nuclear + Hydro)",
    template="plotly_white",
    height=650,
)
fig_treemap.update_traces(
    textinfo="label+value",
    texttemplate="%{label}<br>%{value:,.0f} TWh",
    hovertemplate="<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>",
)
fig_treemap.update_layout(font=dict(family="Arial", size=12))
fig_treemap.show()

# Horizontal bar chart: sorted from largest to smallest.
fig_bar = px.bar(
    low_carbon_by_country.sort_values("TWh", ascending=True),
    x="TWh",
    y="Country",
    orientation="h",
    color_discrete_sequence=["#2E75B6"],
    text="TWh",
    labels={"TWh": "Low-carbon energy (TWh)", "Country": ""},
    title=f"{leading_country} leads low-carbon energy production ({leading_twh:,.0f} TWh)",
    template="plotly_white",
    height=650,
)
fig_bar.update_traces(
    texttemplate="%{text:,.0f} TWh",
    textposition="outside",
    cliponaxis=False,
)
fig_bar.update_layout(
    font=dict(family="Arial", size=12),
    xaxis_title="Low-carbon energy (TWh)",
    yaxis_title="",
    showlegend=False,
)
fig_bar.show()

print(f"Leading country: {leading_country} ({leading_twh:,.1f} TWh)")

Leading country: France (7,593.0 TWh)


Task 2 — reflection
Which chart tells the ranking story more clearly, and why?

The horizontal bar chart tells the ranking story more clearly because all bars share a common baseline, so differences in TWh are easier to compare precisely. Sorting the bars from largest to smallest makes the order immediately visible. The treemap is better for showing relative magnitude and part-to-whole structure at a glance, but comparing rectangle areas precisely is harder.

Task 3 — 100% stacked bar: which region relies most on coal?
What to build: A horizontal 100% stacked bar chart showing, for each Region, how its fossil energy splits between Coal, Oil and Natural Gas. This is the same data as your Task 1 treemap, visualised a second way.

Requirements:

Reuse df_fossil from Task 1; aggregate TWh by Region × Source and convert to shares of each region's fossil total
Horizontal bars, regions sorted by coal share (highest at the top)
Coal is the segment on the baseline (first in category_orders) and the only coloured segment — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
Percentage labels on the coal segments only
Insight title naming the most coal-dependent region and its coal share
💡 df.groupby('Region')['TWh'].transform('sum') gives each row its region total

In [4]:
# Task 3 — 100% stacked horizontal bar: fossil mix by region
# Reuse df_fossil created in Task 1.
region_mix = (
    df_fossil.groupby(["Region", "Source"], as_index=False)["TWh"]
    .sum()
)

# Convert each source's TWh to a share of its region's fossil-energy total.
region_mix["Region_Total"] = region_mix.groupby("Region")["TWh"].transform("sum")
region_mix["Share"] = region_mix["TWh"] / region_mix["Region_Total"]

# Coal share determines the ranking; highest coal share appears at the top.
coal_ranking = (
    region_mix.loc[region_mix["Source"] == "Coal", ["Region", "Share"]]
    .sort_values("Share", ascending=False)
)
region_order = coal_ranking["Region"].tolist()
top_coal_region = coal_ranking.iloc[0]["Region"]
top_coal_share = coal_ranking.iloc[0]["Share"]

# Direct percentage labels on Coal only.
region_mix["Coal_Label"] = np.where(
    region_mix["Source"] == "Coal",
    region_mix["Share"].map("{:.0%}".format),
    "",
)

# Coal uses the same blue as Task 1; Oil and Natural Gas are grey context.
region_colors = {
    "Coal": "#2E75B6",
    "Oil": "#AAAAAA",
    "Natural Gas": "#DDDDDD",
}

fig = px.bar(
    region_mix,
    x="Share",
    y="Region",
    color="Source",
    text="Coal_Label",
    orientation="h",
    color_discrete_map=region_colors,
    category_orders={
        "Region": region_order,
        "Source": ["Coal", "Oil", "Natural Gas"],
    },
    labels={"Share": "Share of regional fossil energy", "Source": "Source"},
    title=(
        f"{top_coal_region} is the most coal-dependent region "
        f"({top_coal_share:.1%} of its fossil-energy total)"
    ),
    template="plotly_white",
    height=600,
)

fig.update_traces(
    textposition="inside",
    insidetextanchor="start",
    textfont=dict(color="white", size=11),
    marker_line_color="white",
    marker_line_width=1,
    hovertemplate="<b>%{y}</b><br>%{fullData.name}: %{x:.1%}<extra></extra>",
)
fig.update_layout(
    font=dict(family="Arial", size=12),
    barmode="stack",
    bargap=0.25,
    xaxis=dict(tickformat=".0%", range=[0, 1], title="Share of regional fossil energy"),
    yaxis_title="",
    legend=dict(orientation="h", x=0, y=1.08),
)
fig.show()

print(f"Most coal-dependent region: {top_coal_region} ({top_coal_share:.1%})")

Most coal-dependent region: Africa (78.0%)


Task 3 — reflection
Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.

Treemap: Which individual country and fossil source contribute the largest absolute amount of fossil energy, and how are countries nested within regions?
100% stacked bar: Which region has the highest coal share of its own fossil-energy mix, regardless of the region's total fossil-energy volume?